# Nanopore3 in Jupyter or Colab

Run **Setup**, then choose **A — Synthetic example** or **B — Your own data**.
Both use the installed CLI through the notebook kernel's Python interpreter.
Failed commands stop their cell; each new analysis gets a fresh output directory.

This notebook also runs locally on Linux, macOS and Windows. Colab-specific Drive
mounting is optional. Colab CPU/RAM allocations vary; the resource cell reports
the effective allocation. Stage inputs and outputs on local disk and allow space
for temporary disk-backed grouping. Measure your own workload before large runs.

## Setup


In [ ]:
# Install once into this kernel's interpreter. Prefer a nearby checkout so local
# fixes are included. Without one, use Git; replace @main with a pinned revision.
import subprocess
import sys
from pathlib import Path

# Set a wheel path or pinned package URL here to override automatic selection.
# Example: r"C:\path\nanopore3-0.3.0-py3-none-any.whl[report,mcp]"
PACKAGE_SPEC = None

if PACKAGE_SPEC is not None:
    install_arguments = [PACKAGE_SPEC]
else:
    candidates = (Path.cwd(), Path.cwd().parent,
                  Path.cwd() / "Nanopore3", Path.cwd() / "nanopore3")
    checkout = next((p for p in candidates
                     if (p / "pyproject.toml").is_file() and (p / "src/nanopore3").is_dir()), None)
    install_arguments = (["-e", f"{checkout}[report,mcp]"] if checkout else
                         ["nanopore3[report,mcp] @ git+https://github.com/t-j-fryer/nanopore3.git@main"])
print("Installing:", install_arguments)
subprocess.run([sys.executable, "-m", "pip", "install", *install_arguments], check=True)
# Restart the kernel after replacing an already imported Nanopore3 installation.
# report supplies pandas/figures; mcp supplies the optional AI connection below.


If the repository is private, install a wheel from your checkout or authenticate
Git through your normal environment. In Colab, mount Drive using the optional
mount cell before installing a wheel stored there. For code that is not yet
published, build/install that checkout's wheel; a branch URL cannot include local
edits. Record the exact wheel/revision and dependency versions for reproducibility.

All CLI calls below use `subprocess.run(..., check=True)` with argument lists,
which works with spaces in Linux/macOS/Windows paths. No shell quoting is needed.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from uuid import uuid4

import pandas as pd
import yaml

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Override NANOPORE3_WORKSPACE before this cell to choose another local disk.
base = Path("/content") if IN_COLAB else Path.cwd()
WORKSPACE = Path(os.environ.get("NANOPORE3_WORKSPACE", str(base / "nanopore3-work"))).expanduser().resolve()
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE / ".cache" / "matplotlib"))

def np3(*arguments, capture=False):
    result = subprocess.run(
        [sys.executable, "-m", "nanopore3", *map(str, arguments)],
        check=True, text=True, stdout=subprocess.PIPE if capture else None,
    )
    return result.stdout if capture else result

print("Local workspace:", WORKSPACE)

In [ ]:
runtime = json.loads(np3("doctor", "--json", capture=True))
print(f"CPUs: {runtime['available_cpus']} allocated / {runtime['cpu_count']} host")
print("Parallel defaults:", runtime["parallel_defaults"])
print(f"Default group estimate budget: {runtime['group_memory_limit_bytes'] / 2**20:g} MiB")
print("This is not a total RSS limit; workers, reference indexes and native tools also use RAM.")
print("Source identity:", runtime["analysis_implementation"]["sha256"])
print(f"Local disk free: {shutil.disk_usage(WORKSPACE).free / 2**30:.1f} GiB")

`jobs: 0` uses all allocated CPUs, including visible Linux affinity/cgroup limits.
Workers and native threads are clamped to that budget. `group_memory_mb` defaults
to 512 MiB and is additionally constrained by visible cgroup memory limits.
It guards estimated retained data for each consensus group/chimera well, not total
RSS. Oversized groups fail clearly without silently dropping reads.

Missing optional `mafft`, `spoa` or `minimap2` is expected with the portable backend.
Use an explicit smaller worker/chunk count if worker copies or reference indexes
need too much RAM. The defaults continue to use maximum allocated parallelism.

---
## A — Try it on the synthetic example


In [ ]:
example = WORKSPACE / f"example-{uuid4().hex[:8]}"
np3("init", example)
demo_id = f"demo-{uuid4().hex[:8]}"
demo_run = WORKSPACE / "runs" / demo_id
np3("run", "--config", example / "configs" / "example.yaml",
    "--output", WORKSPACE / "runs", "--run-id", demo_id)
print("Completed example:", demo_run)

Each stage reports progress. A nonzero exit raises an exception before the cell
can claim completion. Running the cell again creates a new project/run, so a
previous result cannot disguise a failed command.

In [ ]:
clones = pd.read_csv(demo_run / "consensus_by_plate" / "index.csv")
print(clones["grade"].value_counts().to_string())
clones.head()

---
## B — Your own data

Keep inputs, temporary grouping storage and active run outputs on local disk.
For Drive inputs, copy once before repeated input scans; a subsample reads only
the requested prefix. Allow enough disk for uncompressed SQLite grouping data.
Archive completed results to durable storage afterwards.

The optional Drive cell below is only for Colab. In local Jupyter set the input,
reference and archive paths to local directories or mounted storage.

In [ ]:
# This cell can also run before Setup when installing a wheel stored on Drive.
try:
    from google.colab import drive
except ImportError:
    print("Local Jupyter: use your filesystem paths; Drive mounting is optional.")
else:
    drive.mount("/content/drive")


### Point the configuration at this machine

Configurations use `${VARIABLE}` for anything machine-specific, so the same file works
here and on a workstation. Set them for this session:


In [ ]:
# Adjust these to your filesystem or mounted Drive layout.
# Examples on Colab: /content/drive/MyDrive/sequencing and .../references
data_root = Path("/content/drive/MyDrive") if IN_COLAB else Path.home()
os.environ["SEQ_DATA"] = str(data_root / "sequencing")
os.environ["REFS"] = str(data_root / "references")
for name in ("SEQ_DATA", "REFS"):
    where = Path(os.environ[name]).expanduser()
    print(f"{name}: {where} exists={where.is_dir()}")
    assert where.is_dir(), f"Set {name} to an existing directory above"

### Copy locally, or take a validated FASTQ prefix

Set `SUBSAMPLE` to a positive read count or `None` for the entire input. The local
path is new on every execution. Subsampling refuses existing files and aliases,
validates before atomic publication, and requires hard-link support. Use VM-local
disk in Colab, not Drive. A failed command stops before a config can use its output.

In [ ]:
SOURCE = Path(os.environ["SEQ_DATA"]) / "my_run.fastq"  # .fastq.gz also works
SUBSAMPLE = 100_000  # None for the whole file
assert SOURCE.is_file(), f"Not found: {SOURCE}"
if SUBSAMPLE is not None and (isinstance(SUBSAMPLE, bool) or not isinstance(SUBSAMPLE, int) or SUBSAMPLE < 1):
    raise ValueError("SUBSAMPLE must be a positive integer or None")
staging = Path(tempfile.mkdtemp(prefix="input-", dir=WORKSPACE))
local = staging / ("input.fastq.gz" if SOURCE.suffix == ".gz" else "input.fastq")
started = time.monotonic()
if SUBSAMPLE is not None:
    np3("subsample", "--input", SOURCE, "--output", local, "--reads", SUBSAMPLE)
else:
    shutil.copy2(SOURCE, local)
print(f"Input ready: {local} ({time.monotonic() - started:.1f}s)")

### Write the configuration

This example analyses designed inserts with one culture plate per barcode.
Edit motifs, references and barcode registry names to match the experiment; see
[worked examples](https://github.com/t-j-fryer/nanopore3/blob/main/docs/workflows.md).

`preset: ont-r10-amplicon` supplies tuning defaults. Resource choices below are
explicit: all allocated CPUs, one thread per worker, and a 512 MiB group estimate
guard. The existing preset's 300-read consensus cap is retained. Set
`MAXIMUM_READS = 0` to include every eligible read, including chimera members.
Unlimited groups can reach the guard; it never silently reduces their depth.

In [ ]:
JOBS = 0  # all allocated CPUs
GROUP_MEMORY_MB = 512  # estimated retained group data, not total RSS
MAXIMUM_READS = 300  # existing preset cap; 0 means all eligible reads

body = yaml.safe_load("""schema_version: 1
preset: ont-r10-amplicon
run_name: colab-run
output_root: runs

inputs:
  - path: input.fastq
    sample_id: run1

reference_libraries:
  designs:
    fasta: ${REFS}/my_designs.fasta

plate_reference_map:
  BC01: designs

library:
  name: my-amplicon
  forward_motif: CATAATCCGCACGCATCTGG
  reverse_motif: CGGATTGGCGAATGGGACGC
  minimum_read_length: 300
  maximum_read_length: 1200

barcodes:
  plate:
    registry_csv: ${REFS}/plate_barcodes.csv
    family_id: my_plates
  well:
    registry_csv: ${REFS}/well_barcodes.csv
    family_id: my_wells

parallel:
  jobs: 0

random_seed: 1
""")
body["output_root"] = str(WORKSPACE / "runs")
body["inputs"][0]["path"] = str(local)
body["parallel"] = {"backend": "process", "jobs": JOBS, "threads_per_job": 1,
                    "group_memory_mb": GROUP_MEMORY_MB}
body["consensus"] = {"maximum_reads": MAXIMUM_READS}
config = WORKSPACE / "configs" / f"run-{uuid4().hex[:8]}.yaml"
config.parent.mkdir(parents=True, exist_ok=True)
with config.open("x", encoding="utf-8") as handle:
    yaml.safe_dump(body, handle, sort_keys=False)
print(config)
print(config.read_text(encoding="utf-8"))

### Check it before spending the time

`validate` resolves every path, reads the references, checks the barcode panel is
distinguishable and — if you configured pooling — cross-checks the layout against your
reference names. It is much cheaper than finding out at stage five.


In [ ]:
preflight = json.loads(np3("validate", "--config", config, "--quick", capture=True))
print("Resolved CPU plan:", preflight["resources"])
print(f"Resolved group estimate budget: {preflight['group_memory_limit_bytes'] / 2**20:g} MiB")

In [ ]:
run_id = f"analysis-{uuid4().hex[:8]}"
run = WORKSPACE / "runs" / run_id
np3("run", "--config", config, "--output", run.parent, "--run-id", run_id)
print("Completed run:", run)

### Read the results


In [ ]:
clones = pd.read_csv(run / "consensus_by_plate" / "index.csv")
print(clones["grade"].value_counts().to_string())
mixed = clones[clones["grade"].str.startswith("mixed", na=False)]
columns = ["well_id", "design", "grade", "mixed_worst_effect", "designed_allele_fraction"]
mixed[[c for c in columns if c in mixed.columns]] if len(mixed) else "no mixed clones"

### Archive results to durable storage

Use an external/local archive directory or a mounted Drive path. A new directory
is created per export. Summary exports include the complete report stage (with
linked figures), graded clones, QC, source identity and the original YAML.
They do **not** contain all intermediates needed for resume/rerun. Set
`SAVE_FULL_RUN = True` to archive the complete run as well. Only back up completed
or stopped runs so files are not changing while the archive is written.

In [ ]:
archive_base = Path("/content/drive/MyDrive") if IN_COLAB else Path.home()
assert archive_base.is_dir(), "Mount Drive before archiving, or choose durable storage"
ARCHIVE_ROOT = archive_base / "nanopore3-results"  # change to your archive location
SAVE_FULL_RUN = False
OUT = ARCHIVE_ROOT / f"{run.name}-{uuid4().hex[:8]}"
OUT.mkdir(parents=True, exist_ok=False)
if (run / "consensus_by_plate").is_dir():
    shutil.make_archive(str(OUT / "clones"), "zip", run / "consensus_by_plate")
shutil.copytree(run / "stages" / "06_report", OUT / "report")
shutil.copy2(run / "stages" / "05_qc" / "qc.csv.gz", OUT / "qc.csv.gz")
shutil.copy2(run / "run.json", OUT / "run.json")
shutil.copy2(config, OUT / "config.yaml")
if SAVE_FULL_RUN:
    shutil.make_archive(str(OUT / "full-run"), "zip", run.parent, run.name)
print("Archived:", OUT)

---
## Resume or rerun after an interruption

Local Colab files disappear when the VM is recycled. A browser disconnect alone
does not prove the process stopped; check before launching another run. Restore a
full backup onto local disk if needed. Summary exports are insufficient for reuse.
The export cell above handles completed runs. For an interrupted run, stop/check
the process first and archive its whole directory without requiring a report:

```python
# Choose durable storage; for local Jupyter use your own archive directory.
backup_root = Path("/content/drive/MyDrive/nanopore3_results")
assert backup_root.parent.is_dir(), "Mount Drive or choose an existing durable location"
backup_root.mkdir(exist_ok=True)
shutil.make_archive(str(backup_root / f"{run.name}-{uuid4().hex[:8]}"),
                    "zip", run.parent, run.name)
```

**Reuse requires matching installed source identity.** Old runs lacking identity
and runs made with changed code remain readable but need a new run from original
inputs. Pin the same wheel/revision and dependency versions; restart the kernel
and MCP server after changing code. Do not edit manifests to bypass checks.

With the same code, configuration and input evidence, resume an existing run:

```python
np3("run", "--config", config, "--output", run.parent,
    "--run-id", run.name, "--resume")
```

To tune analysis settings, write a new config and recompute into a new run:

```python
np3("rerun", "--config", config, "--from-run", run,
    "--from", "04_consensus", "--output", run.parent,
    "--run-id", f"rerun-{uuid4().hex[:8]}", "--verify")
```

Inherited fingerprints must still match. Original FASTQ is unnecessary when both
input-consuming stages are inherited. All required intermediates and references
must be available. Figures can vary with installed fonts; `run.json` records the
source identity, configuration, input checksums, runtime and resource evidence.

## Optional — connect an AI through MCP

The setup cell installs the MCP extra with the same package revision as the CLI.
Do not reinstall a moving branch here: that could change source identity midway
through the notebook. Local desktop clients can use this kernel's interpreter
and workspace in their MCP configuration. In Colab, the AI client must run inside
the VM; a desktop/cloud AI cannot directly access the VM's loopback server.

The server exposes typed tools for validation, safe subsampling, runs, reruns,
job polling and results. `workspace_info` reports default resource limits and
source identity. `run_summary` previews identity compatibility, which alone does
not establish checksum/config compatibility. See the
[MCP guide](https://github.com/t-j-fryer/nanopore3/blob/main/docs/mcp.md).
The connection below only discovers tools. Keep a connection open until its jobs
finish: closing it cancels active jobs. Colab is not a persistent service host.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

parameters = StdioServerParameters(
    command=sys.executable,
    args=["-m", "nanopore3.mcp_server", "--workspace", str(WORKSPACE)],
    env=dict(os.environ),
)
async with stdio_client(parameters) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        print([tool.name for tool in (await session.list_tools()).tools])
        info = await session.call_tool("workspace_info", {})
        if info.isError:
            raise RuntimeError(info.content)
        print(info.structuredContent)
        guide = await session.read_resource("nanopore3://guide")
        print(guide.contents[0].text)
        # Submit and poll jobs here. For example:
        # start_subsample(input_path="input.fastq", output_path="subset.fastq", reads=100000)
        # Do not use a job's output until job_status reports state="succeeded".